## 11.2 — ממוצע, סטיית תקן ושגיאת הממוצע — ולמה 1/√N

שלוש כמויות, שקל לבלבל ביניהן:

- **ממוצע** $\bar{x}$ — האומדן הטוב ביותר לערך ה"אמיתי".
- **סטיית תקן** $s$ — כמה **מדידה בודדת** מתפזרת סביב הממוצע (תכונה של הנתונים עצמם).
- **שגיאת הממוצע** (Standard Error of the Mean, SEM) $\sigma_{\bar{x}} = s/\sqrt{N}$ — כמה **הממוצע עצמו** היה משתנה אם היינו חוזרים על כל הניסוי (כל ה-$N$ מדידות) שוב ושוב.

ה-$1/\sqrt{N}$ הוא בדיוק הסיבה שיש טעם לחזור על מדידה: יותר מדידות לא הופכות את המדידה הבודדת למדויקת יותר (`s` לא בהכרח קטן), אבל הן כן הופכות את **הממוצע** שלכם לאמין יותר.

```{admonition} 🎥 כאן נכנס סרטון השבוע
:class: seealso

**"מרעש לשגיאה: ממוצע, סטיית תקן, ולמה 1/√N"** (כ-6–10 דקות, הקלטת מסך של המחברת עם קול).

<!-- TODO (למפיק/ה): להחליף תא זה בהטמעת הסרטון בפועל, למשל:
<iframe width="100%" height="400" src="VIDEO_URL_HERE" title="שבוע 11 — מרעש לשגיאה: ממוצע, סטיית תקן, ולמה 1/√N" frameborder="0" allowfullscreen></iframe>
-->
```

### דוגמה: חמש שורות קוד, לא נוסחה מוכתבת

נחשב את שלוש הכמויות "בעבודת יד" (מהנוסחאות), עבור מדידות הטווח בזווית 45°, כדי לראות בדיוק מה כל אחת מהן מודדת.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna().reset_index(drop=True)

r45 = df_clean[df_clean["angle_deg"] == 45]["range_measured"].to_numpy()

N = len(r45)
mean = r45.sum() / N
std = np.sqrt(((r45 - mean)**2).sum() / (N - 1))     # ddof=1 - נסביר בהמשך
sem = std / np.sqrt(N)

print(f"N={N}, ממוצע={mean:.3f}, סטיית תקן={std:.3f}, שגיאת הממוצע={sem:.3f}")

### דוגמה: אותו מספר, שלוש דרכים

באותו ערך אפשר להגיע גם דרך NumPy ישירות (סעיף 9.7) וגם דרך `()describe` של pandas (סעיף 10.6) — שלוש דרכים לאותו מספר, וכדאי להכיר את כולן כדי לזהות אותן בקוד של מישהו אחר.

In [ ]:
# דרך 2: NumPy ישירות (עם ddof=1, כדי להתאים לסטיית תקן מדגמית)
mean_np, std_np = r45.mean(), r45.std(ddof=1)

# דרך 3: pandas describe
desc = df_clean[df_clean["angle_deg"] == 45]["range_measured"].describe()

print("ידני: ", round(mean, 3), round(std, 3))
print("NumPy:", round(mean_np, 3), round(std_np, 3))
print("pandas describe: mean=", round(desc["mean"], 3), " std=", round(desc["std"], 3))

### באג נפוץ: `ddof` שגוי, או $1/N$ במקום $1/\sqrt{N}$

שני בלבולים נפוצים: (1) `np.std(...)` **בברירת המחדל** משתמש ב-`ddof=0` (חלוקה ב-$N$), בעוד ש-`()pandas.Series.std` וכן ה"סטיית תקן המדגמית" הסטטיסטית משתמשות ב-`ddof=1` (חלוקה ב-$N-1$) — התוצאות שונות מעט, במיוחד ב-$N$ קטן. (2) שגיאת הממוצע היא $s/\sqrt{N}$, **לא** $s/N$ — טעות שגורמת להערכת חסר דרסטית של אי-הוודאות כש-$N$ גדול.

In [ ]:
wrong_sem = std / N          # שגוי - זה לא 1/sqrt(N)
right_sem = std / np.sqrt(N)  # נכון

print("שגוי (s/N):", wrong_sem)
print("נכון (s/sqrt(N)):", right_sem)
print("ההבדל גדל ככל ש-N גדול יותר!")

### נסו בעצמכם

חשבו את שגיאת הממוצע של `v0_measured` בזווית 30°. השוו בעצמכם: אם היו לכם רק 2 מדידות במקום כל 8, האם השגיאה הייתה קטנה או גדולה יותר?

In [ ]:
# v30 = df_clean[df_clean["angle_deg"] == 30]["v0_measured"].to_numpy()
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
v30 = df_clean[df_clean["angle_deg"] == 30]["v0_measured"].to_numpy()
sem_v30 = v30.std(ddof=1) / np.sqrt(len(v30))
print(sem_v30)

sem_v30_first2 = v30[:2].std(ddof=1) / np.sqrt(2)
print(sem_v30_first2)   # בדרך כלל גדול יותר - פחות מדידות, פחות ביטחון בממוצע
```
עם פחות מדידות, $\sqrt{N}$ קטן יותר — ולכן שגיאת הממוצע **גדלה**, גם אם סטיית התקן `s` עצמה נשארת דומה.
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "אם כופלים את מספר המדידות פי 4 (אותה std בקירוב), מה קורה לשגיאת הממוצע?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "נשארת אותו דבר", "correct": False, "feedback": "לא — היא תלויה ב-N."},
            {"answer": "קטנה פי 2 (כי sqrt(4)=2)", "correct": True, "feedback": "נכון — 1/sqrt(N) עם N פי 4 גדול יותר."},
            {"answer": "קטנה פי 4", "correct": False, "feedback": "זה היה נכון אם השגיאה הייתה 1/N, אבל היא 1/sqrt(N)."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

עבור כל חמש הזוויות, חשבו טבלה של N, ממוצע, סטיית תקן ושגיאת הממוצע של `range_measured` — בעזרת `groupby` ו-`agg`, ולא בלולאה ידנית.

In [ ]:
# summary = df_clean.groupby("angle_deg")["range_measured"].agg(["count", "mean", "std"])
# summary["sem"] = ...

`````{admonition} פתרון
:class: dropdown, tip
```python
summary = df_clean.groupby("angle_deg")["range_measured"].agg(["count", "mean", "std"])
summary["sem"] = summary["std"] / np.sqrt(summary["count"])
summary
```
`````